# Topshe – chat model v3 (3B, Kaggle)

No uploads needed. This notebook makes its own training data.

**Settings (right panel → Session options):**
- Accelerator: **GPU T4 x2** (or GPU P100)
- Internet: **On** (needs a phone-verified Kaggle account)

**Run it in the background (best):** click **Save Version → Save & Run All (Commit) → Save**. You can close the tab.
It takes about 1.5–2 hours (3B is bigger). When it finishes, open the version → **Output** → download **`topshe_chat_v3.gguf`**.

Outputs: `topshe_chat_v3.gguf` (phone chat model, ~2 GB), `topshe_chat_v3_lora/` (backup), `topshe_chat_v3_eval.txt` (test scores).


In [ ]:
# 1. Setup – use one GPU only (Unsloth works best on a single GPU)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
!pip install -q unsloth
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 2. Make the training data
import json
import random

random.seed(42)

CMD_SYSTEM = (
    "You are Topshe, a phone assistant. Turn the user's request into ONE JSON object. "
    "Actions: set_alarm, set_timer, set_reminder, open_app, call, message, tasker, stop, chat. "
    "Use chat for questions or small talk. Reply with JSON only."
)

# Add your own apps and contacts here, then re-run
APPS = ["WhatsApp", "YouTube", "Spotify", "Chrome", "Gmail", "Maps", "Camera", "Instagram",
        "Telegram", "Settings", "Calculator", "Calendar", "Kindle", "Zomato", "Swiggy", "Uber",
        "Ola", "Google Pay", "PhonePe", "Notes", "Files", "Clock", "Photos", "Zerodha Kite",
        "Netflix", "Hotstar", "Drive", "Meet", "Zoom", "Slack", "LinkedIn", "Twitter",
        "Blinkit", "Zepto", "Amazon", "Flipkart", "Paytm", "Outlook", "Teams", "Discord"]
CONTACTS = ["Maa", "Baba", "Tanmay", "Rahul", "Priya", "Didi", "Dada", "Arjun", "Sneha",
            "Professor Sen", "Ankit", "Riya", "Mama", "Kaku", "Shreya", "Sourav", "Ritwik",
            "Pishi", "Jethu", "Anirban", "Debjani", "Rohan", "Ishita", "Sayan", "Moumita"]
MSG_TEXTS = ["I'll be late", "reached home", "call me when free", "on my way",
             "running 10 minutes late", "can we meet tomorrow", "sent the file",
             "happy birthday", "I'm in class right now", "dinner at 9?", "the draft is ready",
             "I'm taking the shuttle", "will call you later", "check your email",
             "ok see you soon", "I'm in a meeting"]
TASKS = ["wifi", "bluetooth", "silent mode", "dnd", "torch", "battery saver", "hotspot",
         "dark mode", "study mode", "mobile data", "location", "airplane mode"]
REMIND_TEXTS = ["submit the assignment", "call Maa", "take the shuttle", "RA meeting",
                "buy milk", "pay the electricity bill", "check email", "drink water",
                "start the Stata run", "read the paper", "take my medicine", "book the cab",
                "charge the laptop", "reply to the professor", "go to the gym"]
CHAT_Q = ["What is the capital of Japan?", "Who wrote Sonar Kella?", "How far is the moon?",
          "Tell me a joke", "How are you?", "What is 17 times 23?", "Who is Feluda?",
          "What should I cook tonight?", "How many litres in a gallon?", "Give me a fun fact",
          "Translate thank you to Bengali", "Thank you Topshe", "Suggest a good book",
          "Why is the sky blue?", "What's a p-value?", "Good morning", "What can you do?",
          "Who are you?", "How many days in a leap year?", "What's 15 percent of 240?"]
TOPICS = ["black holes", "the Mughal empire", "Satyajit Ray", "photosynthesis", "inflation",
          "machine learning", "the stock market", "Durga Puja", "the Bengal famine", "cricket",
          "climate change", "the RBI", "vaccines", "the Taj Mahal", "Rabindranath Tagore",
          "game theory", "the French Revolution", "monsoons", "GST", "quantum computing",
          "the Sundarbans", "regression", "Netaji Subhas Chandra Bose", "the IPL", "DNA",
          "instrumental variables", "mutual funds", "the Indian constitution", "volcanoes"]
CHAT_TMPL = ["what is {x}", "tell me about {x}", "explain {x} simply", "who started {x}",
             "why is {x} important", "give me a fact about {x}", "what do you know about {x}",
             "{x} ke baare mein batao", "{x} niye kichu bolo", "summarise {x} in two lines",
             "how does {x} work", "is {x} a good thing"]


def fmt_time(h, m):
    ampm = "am" if h < 12 else "pm"
    h12 = h % 12 or 12
    opts = [f"{h12}:{m:02d} {ampm}", f"{h12}:{m:02d}{ampm}", f"{h12} {m:02d} {ampm}"]
    if m == 0:
        opts += [f"{h12} {ampm}", f"{h12} o'clock " + ("in the morning" if h < 12 else "in the evening")]
    if m == 30:
        opts.append(f"half past {h12} {ampm}")
    if m == 15:
        opts.append(f"quarter past {h12} {ampm}")
    return random.choice(opts)


def ex_alarm():
    h = random.randint(0, 23)
    m = random.choice([0, 0, 0, 15, 30, 30, 45, random.randint(0, 59)])
    tmpl = random.choice([
        "wake me up at {t}", "set an alarm for {t}", "alarm at {t}", "set alarm {t} tomorrow",
        "I need to get up at {t} tomorrow", "kal subah {t} baje alarm laga do", "{t} er alarm dao",
        "kal {t} e alarm dao", "alarm for {t} today", "put an alarm at {t}",
        "set a {t} alarm", "{t} pe alarm set karo",
    ])
    day = "tomorrow" if ("tomorrow" in tmpl or "kal" in tmpl) else ("today" if "today" in tmpl else "next")
    return tmpl.format(t=fmt_time(h, m)), {"action": "set_alarm", "hour": h, "minute": m, "day": day}


def ex_timer():
    unit = random.choice(["seconds", "minutes", "minutes", "minutes", "hours"])
    n = random.randint(1, 3) if unit == "hours" else random.randint(1, 90)
    word = unit[:-1] if n == 1 else unit
    secs = n * {"seconds": 1, "minutes": 60, "hours": 3600}[unit]
    text = random.choice([
        "set a timer for {n} {u}", "timer {n} {u}", "start a {n} {u} timer", "{n} {u} ka timer lagao",
        "count down {n} {u}", "{n} {u} er timer dao", "time {n} {u}",
    ]).format(n=n, u=word)
    return text, {"action": "set_timer", "seconds": secs}


def ex_reminder():
    what = random.choice(REMIND_TEXTS)
    if random.random() < 0.5:
        n = random.choice([5, 10, 15, 20, 30, 45, 60, 90, 120])
        text = random.choice(["remind me to {w} in {n} minutes", "in {n} minutes remind me to {w}",
                              "{n} minute baad yaad dilana {w}", "remind me in {n} minutes to {w}"]).format(w=what, n=n)
        return text, {"action": "set_reminder", "text": what, "in_minutes": n}
    h, m = random.randint(6, 23), random.choice([0, 15, 30, 45])
    day = random.choice(["today", "tomorrow"])
    text = random.choice(["remind me to {w} at {t} {d}", "{d} at {t} remind me to {w}",
                          "set a reminder to {w} at {t} {d}", "remind me {d} at {t} to {w}"]).format(w=what, t=fmt_time(h, m), d=day)
    return text, {"action": "set_reminder", "text": what, "hour": h, "minute": m, "day": day}


def ex_open():
    app = random.choice(APPS)
    text = random.choice(["open {a}", "launch {a}", "start {a}", "{a} kholo", "go to {a}",
                          "can you open {a}", "{a} khulo", "open up {a}", "take me to {a}"]).format(a=app)
    return text, {"action": "open_app", "app": app}


def ex_call():
    c = random.choice(CONTACTS)
    text = random.choice(["call {c}", "phone {c}", "ring {c}", "{c} ko call karo", "{c} ke phone koro",
                          "give {c} a call", "dial {c}", "call {c} now", "make a call to {c}"]).format(c=c)
    return text, {"action": "call", "contact": c}


def ex_message():
    c, msg = random.choice(CONTACTS), random.choice(MSG_TEXTS)
    if random.random() < 0.3:
        app = "SMS"
        text = random.choice(["send {c} an SMS saying {m}", "SMS {c} {m}", "text {c} by SMS that {m}"]).format(c=c, m=msg)
    elif random.random() < 0.5:
        app = "WhatsApp"
        text = random.choice(["whatsapp {c} {m}", "send {c} a whatsapp saying {m}",
                              "tell {c} on whatsapp that {m}"]).format(c=c, m=msg)
    else:
        app = "WhatsApp"  # default when no app is named
        text = random.choice(["message {c} {m}", "text {c} saying {m}", "tell {c} {m}",
                              "{c} ko message bhejo {m}", "send a message to {c} that {m}"]).format(c=c, m=msg)
    return text, {"action": "message", "app": app, "contact": c, "text": msg}


def ex_tasker():
    t, state = random.choice(TASKS), random.choice(["on", "off"])
    text = random.choice(["turn {s} {t}", "turn {t} {s}", "switch {s} {t}", "switch {t} {s}",
                          "{t} {s}", "{t} {s} karo", "{t} {s} koro", "put {t} {s}"]).format(t=t, s=state)
    return text, {"action": "tasker", "task": f"{t} {state}"}


def ex_stop():
    text = random.choice(["stop listening", "go to sleep", "topshe off", "turn yourself off",
                          "that's all", "bye topshe", "stop", "chup karo", "thamo", "you can sleep now"])
    return text, {"action": "stop"}


def ex_chat():
    q = random.choice(CHAT_Q) if random.random() < 0.3 else random.choice(CHAT_TMPL).format(x=random.choice(TOPICS))
    return q, {"action": "chat", "query": q}


GENERATORS = [(ex_alarm, 700), (ex_timer, 400), (ex_reminder, 700), (ex_open, 500), (ex_call, 400),
              (ex_message, 700), (ex_tasker, 400), (ex_stop, 100), (ex_chat, 700)]


def noisy(text):
    """Make text look like speech-to-text output."""
    if random.random() < 0.15:
        text = random.choice(["topshe ", "hey topshe ", "please ", "umm ", "ok "]) + text
    if random.random() < 0.1:
        text += random.choice([" please", " thanks", " now"])
    text = text.replace("?", "") if random.random() < 0.5 else text
    return text.lower() if random.random() < 0.6 else text


def make_rows():
    rows, seen = [], set()
    for gen, n in GENERATORS:
        made = tries = 0
        while made < n and tries < n * 30:
            tries += 1
            text, out = gen()
            text = " ".join(noisy(text).split())
            if text in seen:
                continue
            seen.add(text)
            rows.append({"messages": [
                {"role": "system", "content": CMD_SYSTEM},
                {"role": "user", "content": text},
                {"role": "assistant", "content": json.dumps(out, ensure_ascii=False)},
            ]})
            made += 1
    random.shuffle(rows)
    return rows

rows = make_rows()
random.seed(7)
random.shuffle(rows)
val_rows, train_cmd = rows[:200], rows[200:]
with open("/kaggle/working/topshe_commands_v3.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(train_cmd), "train commands,", len(val_rows), "held back for testing")


In [ ]:
# 3. Load the base model
from unsloth import FastLanguageModel
import torch

MAX_LEN = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)


In [ ]:
# 4. General chat examples (more than v2: this model is for conversation)
from datasets import load_dataset, Dataset

CHAT_SYSTEM = "You are Topshe, a friendly assistant. Answer in 1-3 short, simple sentences. Talk to Annay plainly, like an adult. No lectures or disclaimers."
alpaca = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=42)
chat = []
for r in alpaca:
    if r["input"] or len(r["output"]) > 300 or len(r["output"]) < 5:
        continue
    chat.append({"messages": [
        {"role": "system", "content": CHAT_SYSTEM},
        {"role": "user", "content": r["instruction"]},
        {"role": "assistant", "content": r["output"]},
    ]})
    if len(chat) >= 4000:
        break

train_rows = train_cmd + chat
random.shuffle(train_rows)
texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in train_rows]
ds = Dataset.from_dict({"text": texts})
print(len(train_cmd), "commands +", len(chat), "chat =", len(ds), "training rows")


In [ ]:
# 5. Train (about 60-90 min)
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

cfg = dict(
    dataset_text_field="text",
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=2e-4, warmup_steps=20,
    logging_steps=25, optim="adamw_8bit", weight_decay=0.01,
    lr_scheduler_type="linear", seed=42, output_dir="/kaggle/tmp_outputs",
    report_to="none", save_strategy="no",
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
)
# TRL renamed max_seq_length -> max_length in newer versions
params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in params else "max_seq_length"] = MAX_LEN
args = SFTConfig(**cfg)

try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)

# Learn only from Topshe's replies, not from the prompts
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

# Backup: save the small LoRA adapter first, so nothing is lost if export fails
model.save_pretrained("/kaggle/working/topshe_chat_v3_lora")
tokenizer.save_pretrained("/kaggle/working/topshe_chat_v3_lora")
print("Saved LoRA backup")


In [ ]:
# 6. Test on the 200 held-back commands
FastLanguageModel.for_inference(model)

def ask(text, system=CMD_SYSTEM, max_new=100):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

valid_json = right_action = exact = 0
mistakes = []
for r in val_rows:
    text, truth = r["messages"][1]["content"], json.loads(r["messages"][2]["content"])
    pred_raw = ask(text)
    try:
        pred = json.loads(pred_raw)
        valid_json += 1
    except Exception:
        mistakes.append((text, pred_raw, truth)); continue
    right_action += pred.get("action") == truth["action"]
    if pred == truth:
        exact += 1
    else:
        mistakes.append((text, pred_raw, truth))

n = len(val_rows)
report = (f"Valid JSON: {valid_json}/{n}\nRight action: {right_action}/{n}\n"
          f"Fully correct: {exact}/{n}\n\nSome mistakes:\n" +
          "\n".join(f"- {t}\n    got:  {p}\n    want: {json.dumps(w)}" for t, p, w in mistakes[:15]))
print(report)

print("\n--- Fresh examples ---")
for t in ["wake me up at 6:45 tomorrow", "remind me to call baba in 25 minutes",
          "open spotify", "message tanmay that the draft is ready", "turn wifi off",
          "topshe go to sleep", "who directed pather panchali"]:
    print(t, "=>", ask(t))
chat_reply = ask("who directed pather panchali", CHAT_SYSTEM, 80)
print("\nChat mode:", chat_reply)

open("/kaggle/working/topshe_chat_v3_eval.txt", "w").write(report + "\n\nChat test: " + chat_reply)


In [ ]:
# 7. Export the phone model (GGUF)
import glob, shutil, subprocess

OUT = "/kaggle/working/topshe_chat_v3.gguf"
done = False

# Way 1: Unsloth's built-in export, 4-bit (~1 GB)
try:
    model.save_pretrained_gguf("/kaggle/tmp_gguf", tokenizer, quantization_method="q4_k_m")
    found = [p for p in glob.glob("/kaggle/tmp_gguf*/**/*.gguf", recursive=True) + glob.glob("/kaggle/tmp_gguf*.gguf")
             if "q4_k_m" in p.lower()]
    shutil.copy(found[0], OUT)
    done = True
    print("Exported with Unsloth (Q4_K_M)")
except Exception as e:
    print("Unsloth export failed, trying the backup way:", repr(e)[:300])

# Way 2 (backup): merge, then convert with llama.cpp's script, 8-bit (~1.6 GB, no compiling needed)
if not done:
    model.save_pretrained_merged("/kaggle/tmp_merged", tokenizer, save_method="merged_16bit")
    subprocess.run("git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/tmp_llama", shell=True, check=True)
    subprocess.run("pip install -q -r /kaggle/tmp_llama/requirements/requirements-convert_hf_to_gguf.txt", shell=True, check=True)
    subprocess.run(f"python /kaggle/tmp_llama/convert_hf_to_gguf.py /kaggle/tmp_merged --outfile {OUT} --outtype q8_0",
                   shell=True, check=True)
    print("Exported with llama.cpp (Q8_0)")

print(OUT, round(os.path.getsize(OUT) / 1e9, 2), "GB")


In [ ]:
# 8. Files ready to download (Output panel)
for p in glob.glob("/kaggle/working/*"):
    print(os.path.basename(p))
